# 01 — Data Exploration

Explore the CICIDS2017 (or sample) dataset: shape, label distribution,
feature statistics, and class-wise feature comparisons.

In [ ]:
import os, sys
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

DATA_PATH = os.path.join('..', 'data', 'sample_data.csv')
df = pd.read_csv(DATA_PATH)
df.columns = df.columns.str.strip()
print('Shape:', df.shape)
print(df.dtypes.value_counts())

## Label distribution

In [ ]:
label_counts = df['Label'].value_counts()
print(label_counts)

fig, ax = plt.subplots(figsize=(7, 4))
label_counts.plot(kind='bar', ax=ax, color=['#2ecc71','#e74c3c','#3498db'], edgecolor='black')
ax.set_title('Label Distribution — CICIDS2017 Sample')
ax.set_xlabel('Class')
ax.set_ylabel('Count')
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig(os.path.join('..', 'results', 'plots', 'label_distribution.png'), dpi=120)
plt.show()

## Feature statistics

In [ ]:
# Create binary label for display
df_display = df.copy()
df_display['attack'] = (df_display['Label'] != 'BENIGN').astype(int)

num_cols = [c for c in df.columns if c != 'Label']
df[num_cols].describe().round(1)

## Class-wise feature comparison (top 6 discriminative features)

In [ ]:
top_feats = ['FlowBytess', 'FlowPacketss', 'FlowDuration',
             'TotalFwdPackets', 'FlowIATMean', 'FwdPacketLengthMean']
# Use available columns
top_feats = [f for f in top_feats if f in df.columns][:6]

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

for i, feat in enumerate(top_feats):
    benign_vals = df[df['Label']=='BENIGN'][feat].clip(upper=df[feat].quantile(0.99))
    attack_vals = df[df['Label']!='BENIGN'][feat].clip(upper=df[feat].quantile(0.99))
    axes[i].hist(benign_vals, bins=40, alpha=0.6, color='#2ecc71', label='BENIGN')
    axes[i].hist(attack_vals, bins=40, alpha=0.6, color='#e74c3c', label='ATTACK')
    axes[i].set_title(feat, fontsize=10)
    axes[i].legend(fontsize=8)
    axes[i].set_xlabel('Value')

plt.suptitle('Feature Distributions: BENIGN vs ATTACK', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(os.path.join('..', 'results', 'plots', 'feature_distributions.png'), dpi=120, bbox_inches='tight')
plt.show()

## Correlation heatmap

In [ ]:
selected = [c for c in ['FlowBytess','FlowPacketss','FlowDuration',
            'TotalFwdPackets','TotalBackwardPackets','FlowIATMean',
            'FwdPacketLengthMean','BwdPacketLengthMean','SYNFlagCount',
            'PSHFlagCount'] if c in df.columns]

corr_df = df[selected].copy()
corr_df['IsAttack'] = (df['Label'] != 'BENIGN').astype(int)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr_df.corr(), annot=True, fmt='.2f', cmap='coolwarm',
            linewidths=0.4, ax=ax, center=0)
ax.set_title('Feature Correlation Matrix')
plt.tight_layout()
plt.savefig(os.path.join('..', 'results', 'plots', 'correlation_heatmap.png'), dpi=120, bbox_inches='tight')
plt.show()